In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk
import sys
sys.path.append("../")

DATA_DIR = "../mu3e_trigger_data"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_with_layer_pixel_spacetime.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_with_layer_mppc_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_mppc_spacetime.npy"
SIGNAL_ONLY_PIXEL_FILE = f"{DATA_DIR}/sig_only_with_layer_pixel_spacetime.npy"
SIGNAL_ONLY_MPPC_FILE = f"{DATA_DIR}/sig_only_with_layer_mppc_spacetime.npy"


bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
sig_only_pixel_spacetime = np.load(SIGNAL_ONLY_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)
sig_only_mppc_spacetime = np.load(SIGNAL_ONLY_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GCNConv, Linear, global_mean_pool
from torch_geometric.data import HeteroData, Batch
from torch_geometric.utils import negative_sampling
from typing import Dict, List, Tuple, Optional, Union
from collections import defaultdict


class NodeTypeEncoder(nn.Module):
    """Encoder for a specific node type."""
    
    def __init__(self, input_dim: int, hidden_dims: List[int], output_dim: int):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        dims = [input_dim] + hidden_dims + [output_dim]
        self.layers = nn.ModuleList([
            nn.Linear(dims[i], dims[i+1]) for i in range(len(dims)-1)
        ])
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:  # No activation on last layer
                x = F.relu(x)
                x = self.dropout(x)
        return x


class NodeTypeDecoder(nn.Module):
    """Decoder for a specific node type."""
    
    def __init__(self, input_dim: int, hidden_dims: List[int], output_dim: int):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        dims = [input_dim] + hidden_dims + [output_dim]
        self.layers = nn.ModuleList([
            nn.Linear(dims[i], dims[i+1]) for i in range(len(dims)-1)
        ])
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        for i, layer in enumerate(self.layers):
            z = layer(z)
            if i < len(self.layers) - 1:  # No activation on last layer for reconstruction
                z = F.relu(z)
                z = self.dropout(z)
        return z


class EdgeDecoder(nn.Module):
    """Decoder for edge reconstruction using dot product."""
    
    def __init__(self, latent_dim: int):
        super().__init__()
        self.latent_dim = latent_dim
        
    def forward(self, z_src: torch.Tensor, z_dst: torch.Tensor, 
                edge_index: torch.Tensor) -> torch.Tensor:
        """Decode edges using dot product of node embeddings."""
        row, col = edge_index[0], edge_index[1]
        return (z_src[row] * z_dst[col]).sum(dim=1)


class HeteroGraphEncoder(nn.Module):
    """Heterogeneous graph encoder with separate processing for each node/edge type."""
    
    def __init__(self, 
                 node_dims: Dict[str, int], 
                 latent_dim: int = 64, 
                 hidden_dims: List[int] = [128, 64],
                 num_conv_layers: int = 2):
        super().__init__()
        
        self.node_dims = node_dims
        self.latent_dim = latent_dim
        self.num_conv_layers = num_conv_layers
        
        # Node type encoders - first projection to common space
        self.node_encoders = nn.ModuleDict({
            node_type: NodeTypeEncoder(dim, hidden_dims, latent_dim)
            for node_type, dim in node_dims.items()
        })
        
        # Define edge types for the layer-separated graph
        self.edge_types = [
            ('layer_1', 'to', 'layer_2'),
            ('layer_2', 'to', 'layer_1'),
            ('layer_2', 'to', 'mppc'),
            ('mppc', 'to', 'layer_2'),
            ('mppc', 'to', 'mppc'),
            ('mppc', 'to', 'layer_3'),
            ('layer_3', 'to', 'mppc'),
            ('layer_3', 'to', 'layer_4'),
            ('layer_4', 'to', 'layer_3'),
        ]
        
        # Heterogeneous convolution layers
        self.convs = nn.ModuleList([
            HeteroConv({
                edge_type: GCNConv(latent_dim, latent_dim)
                for edge_type in self.edge_types
            }, aggr='sum') for _ in range(num_conv_layers)
        ])
        
        # Final projection to latent space
        self.final_projection = nn.ModuleDict({
            node_type: nn.Linear(latent_dim, latent_dim)
            for node_type in node_dims.keys()
        })
        
    def forward(self, x_dict: Dict[str, torch.Tensor], 
                edge_index_dict: Dict[Tuple[str, str, str], torch.Tensor]) -> Dict[str, torch.Tensor]:
        """Forward pass through heterogeneous graph encoder."""
        
        # Initial node encoding
        z_dict = {}
        for node_type, x in x_dict.items():
            if node_type in self.node_encoders:
                z_dict[node_type] = self.node_encoders[node_type](x)
        
        # Message passing through heterogeneous convolutions
        for conv in self.convs:
            z_dict = conv(z_dict, edge_index_dict)
            # Apply activation after each conv layer except the last
            z_dict = {key: F.relu(z) for key, z in z_dict.items()}
        
        # Final projection
        for node_type, z in z_dict.items():
            z_dict[node_type] = self.final_projection[node_type](z)
        
        return z_dict


class HeteroGraphDecoder(nn.Module):
    """Heterogeneous graph decoder for reconstructing node features and edges."""
    
    def __init__(self, 
                 node_dims: Dict[str, int], 
                 latent_dim: int = 64, 
                 hidden_dims: List[int] = [64, 128]):
        super().__init__()
        
        self.node_dims = node_dims
        self.latent_dim = latent_dim
        
        # Node type decoders
        self.node_decoders = nn.ModuleDict({
            node_type: NodeTypeDecoder(latent_dim, hidden_dims, dim)
            for node_type, dim in node_dims.items()
        })
        
        # Edge decoders for each edge type
        self.edge_decoders = nn.ModuleDict({
            f"{src}_{rel}_{dst}": EdgeDecoder(latent_dim)
            for src, rel, dst in [
                ('layer_1', 'to', 'layer_2'),
                ('layer_2', 'to', 'layer_1'),
                ('layer_2', 'to', 'mppc'),
                ('mppc', 'to', 'layer_2'),
                ('mppc', 'to', 'mppc'),
                ('mppc', 'to', 'layer_3'),
                ('layer_3', 'to', 'mppc'),
                ('layer_3', 'to', 'layer_4'),
                ('layer_4', 'to', 'layer_3'),
            ]
        })
        
    def forward(self, z_dict: Dict[str, torch.Tensor], 
                edge_index_dict: Dict[Tuple[str, str, str], torch.Tensor]) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
        """Forward pass through decoder."""
        
        # Reconstruct node features
        x_recon_dict = {}
        for node_type, z in z_dict.items():
            if node_type in self.node_decoders:
                x_recon_dict[node_type] = self.node_decoders[node_type](z)
        
        # Reconstruct edges
        edge_probs_dict = {}
        for (src, rel, dst), edge_index in edge_index_dict.items():
            if src in z_dict and dst in z_dict and edge_index.size(1) > 0:
                edge_key = f"{src}_{rel}_{dst}"
                if edge_key in self.edge_decoders:
                    edge_probs = torch.sigmoid(
                        self.edge_decoders[edge_key](z_dict[src], z_dict[dst], edge_index)
                    )
                    edge_probs_dict[(src, rel, dst)] = edge_probs
        
        return x_recon_dict, edge_probs_dict


class HeteroGraphAutoencoder(nn.Module):
    """Complete heterogeneous graph autoencoder."""
    
    def __init__(self, 
                 node_dims: Dict[str, int], 
                 latent_dim: int = 64,
                 hidden_dims: List[int] = [128, 64],
                 num_conv_layers: int = 2):
        super().__init__()
        
        self.node_dims = node_dims
        self.latent_dim = latent_dim
        
        self.encoder = HeteroGraphEncoder(
            node_dims, latent_dim, hidden_dims, num_conv_layers
        )
        self.decoder = HeteroGraphDecoder(
            node_dims, latent_dim, list(reversed(hidden_dims))
        )
        
    def forward(self, data: HeteroData) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
        """Full forward pass through autoencoder."""
        # Encode
        z_dict = self.encoder(data.x_dict, data.edge_index_dict)
        
        # Decode
        x_recon_dict, edge_probs_dict = self.decoder(z_dict, data.edge_index_dict)
        
        return x_recon_dict, edge_probs_dict
    
    def encode(self, data: HeteroData) -> Dict[str, torch.Tensor]:
        """Encode graph to latent representation."""
        return self.encoder(data.x_dict, data.edge_index_dict)
    
    def decode(self, z_dict: Dict[str, torch.Tensor], 
               edge_index_dict: Dict[Tuple[str, str, str], torch.Tensor]) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
        """Decode from latent representation."""
        return self.decoder(z_dict, edge_index_dict)


class HeteroGraphAELoss(nn.Module):
    """Loss function for heterogeneous graph autoencoder."""
    
    def __init__(self, 
                 node_weight: float = 1.0, 
                 edge_weight: float = 1.0,
                 neg_sampling_ratio: float = 1.0):
        super().__init__()
        self.node_weight = node_weight
        self.edge_weight = edge_weight
        self.neg_sampling_ratio = neg_sampling_ratio
        self.mse_loss = nn.MSELoss()
        self.bce_loss = nn.BCELoss()
        
    def forward(self, 
                data: HeteroData,
                x_recon_dict: Dict[str, torch.Tensor], 
                edge_probs_dict: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """Calculate reconstruction loss."""
        
        losses = {}
        total_loss = 0.0
        
        # Node reconstruction loss
        node_loss = 0.0
        node_count = 0
        for node_type, x_recon in x_recon_dict.items():
            if node_type in data.x_dict:
                loss = self.mse_loss(x_recon, data.x_dict[node_type])
                node_loss += loss
                node_count += 1
                losses[f'node_{node_type}'] = loss.item()
        
        if node_count > 0:
            node_loss /= node_count
            losses['node_total'] = node_loss.item()
            total_loss += self.node_weight * node_loss
        
        # Edge reconstruction loss
        edge_loss = 0.0
        edge_count = 0
        for (src, rel, dst), edge_probs in edge_probs_dict.items():
            if (src, rel, dst) in data.edge_index_dict:
                edge_index = data.edge_index_dict[(src, rel, dst)]
                if edge_index.size(1) > 0:
                    # Positive edges (actual edges in graph)
                    pos_labels = torch.ones(edge_probs.size(0), device=edge_probs.device)
                    pos_loss = self.bce_loss(edge_probs, pos_labels)
                    
                    # Negative sampling for better edge prediction
                    if src in data.x_dict and dst in data.x_dict:
                        num_nodes_src = data.x_dict[src].size(0)
                        num_nodes_dst = data.x_dict[dst].size(0)
                        num_neg = int(edge_index.size(1) * self.neg_sampling_ratio)
                        
                        if num_neg > 0:
                            neg_edge_index = negative_sampling(
                                edge_index, num_nodes=(num_nodes_src, num_nodes_dst),
                                num_neg_samples=num_neg
                            )
                            
                            # Get negative edge probabilities
                            neg_edge_probs = torch.sigmoid(
                                (data.x_dict[src][neg_edge_index[0]] * 
                                 data.x_dict[dst][neg_edge_index[1]]).sum(dim=1)
                            )
                            neg_labels = torch.zeros(neg_edge_probs.size(0), device=edge_probs.device)
                            neg_loss = self.bce_loss(neg_edge_probs, neg_labels)
                            
                            edge_type_loss = (pos_loss + neg_loss) / 2
                        else:
                            edge_type_loss = pos_loss
                    else:
                        edge_type_loss = pos_loss
                    
                    edge_loss += edge_type_loss
                    edge_count += 1
                    losses[f'edge_{src}_{rel}_{dst}'] = edge_type_loss.item()
        
        if edge_count > 0:
            edge_loss /= edge_count
            losses['edge_total'] = edge_loss.item()
            total_loss += self.edge_weight * edge_loss
        
        losses['total'] = total_loss.item()
        return total_loss, losses


class HeteroGraphAETrainer:
    """Trainer for heterogeneous graph autoencoder."""
    
    def __init__(self, 
                 model: HeteroGraphAutoencoder,
                 learning_rate: float = 0.001,
                 weight_decay: float = 1e-5):
        self.model = model
        self.optimizer = torch.optim.Adam(
            model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )
        self.loss_fn = HeteroGraphAELoss()
        self.device = next(model.parameters()).device
        
    def train_step(self, batch: HeteroData) -> Dict[str, float]:
        """Single training step."""
        self.model.train()
        self.optimizer.zero_grad()
        
        batch = batch.to(self.device)
        x_recon_dict, edge_probs_dict = self.model(batch)
        
        loss, losses = self.loss_fn(batch, x_recon_dict, edge_probs_dict)
        loss.backward()
        self.optimizer.step()
        
        return losses
    
    def validate_step(self, batch: HeteroData) -> Dict[str, float]:
        """Single validation step."""
        self.model.eval()
        with torch.no_grad():
            batch = batch.to(self.device)
            x_recon_dict, edge_probs_dict = self.model(batch)
            loss, losses = self.loss_fn(batch, x_recon_dict, edge_probs_dict)
        
        return losses
    
    def train_epoch(self, dataloader) -> Dict[str, float]:
        """Train for one epoch."""
        epoch_losses = defaultdict(list)
        
        for batch in dataloader:
            losses = self.train_step(batch)
            for key, value in losses.items():
                epoch_losses[key].append(value)
        
        # Average losses
        avg_losses = {key: np.mean(values) for key, values in epoch_losses.items()}
        return avg_losses
    
    def validate_epoch(self, dataloader) -> Dict[str, float]:
        """Validate for one epoch."""
        epoch_losses = defaultdict(list)
        
        for batch in dataloader:
            losses = self.validate_step(batch)
            for key, value in losses.items():
                epoch_losses[key].append(value)
        
        # Average losses
        avg_losses = {key: np.mean(values) for key, values in epoch_losses.items()}
        return avg_losses


def create_hetero_autoencoder_for_layers(latent_dim: int = 64) -> HeteroGraphAutoencoder:
    """Create autoencoder specifically for layer-separated heterogeneous graphs."""
    
    # Define node dimensions for each detector type
    node_dims = {
        'layer_1': 3,  # [x, y, z]
        'layer_2': 3,  # [x, y, z]
        'layer_3': 3,  # [x, y, z]
        'layer_4': 3,  # [x, y, z]
        'mppc': 4,     # [x, y, z, time]
    }
    
    return HeteroGraphAutoencoder(
        node_dims=node_dims,
        latent_dim=latent_dim,
        hidden_dims=[128, 64],
        num_conv_layers=3
    )


def train_hetero_autoencoder(model: HeteroGraphAutoencoder,
                           train_loader,
                           val_loader=None,
                           num_epochs: int = 100,
                           device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
    """Complete training function for the heterogeneous graph autoencoder."""
    
    model = model.to(device)
    trainer = HeteroGraphAETrainer(model)
    
    best_val_loss = float('inf')
    training_history = {
        'train_losses': [],
        'val_losses': []
    }
    
    for epoch in range(num_epochs):
        # Training
        train_losses = trainer.train_epoch(train_loader)
        training_history['train_losses'].append(train_losses)
        
        # Validation
        if val_loader is not None:
            val_losses = trainer.validate_epoch(val_loader)
            training_history['val_losses'].append(val_losses)
            
            if val_losses['total'] < best_val_loss:
                best_val_loss = val_losses['total']
                torch.save(model.state_dict(), 'best_hetero_autoencoder.pth')
        
        # Print progress
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Train Loss: {train_losses['total']:.4f}", end="")
            if val_loader is not None:
                print(f" | Val Loss: {val_losses['total']:.4f}")
            else:
                print()
    
    return model, training_history